# CONSTRUCTION D'UN ASSISTANT IA

> Ce qui paie, ce n'est pas de savoir *utiliser* l'IA. C'est la capacité à **CONSTRUIRE**.

---

Ceci est la **version documentée** du projet, pensée pour comprendre au maximum ce qui s'est passé derrière chaque ligne de code - pas juste le résultat final, mais le raisonnement complet.

En une session avec Machine Learnia, j'ai conçu un assistant IA capable de répondre à des questions à partir de mes propres documents, sans halluciner, avec citation des sources - grâce à la technique du **RAG (Retrieval-Augmented Generation)**.

**Ce que j'ai mis en place :**
- Une base documentaire fictive (Librairie du Savoir - 3 agences + conditions générales)
- Un moteur de recherche sémantique par embeddings (compréhension du sens, pas des mots-clés)
- Un modèle de langage local, contraint à ne répondre qu'à partir des documents fournis
- Une interface web déployée en direct avec Gradio

*Démonstration et détails techniques ci-dessous.*

# JALON 0 : PREPARATION DE L"ENVIRONNEMENT

On installe les outils : le lecteur de PDF, le moteur de recherche sémantique, et l'atelier de mise en ligne.

Trois librairies sont installees : 

- **sentence-transformers : la librairie qui va transformer du texte en vecteurs numeriques (les embeddings), indispensable pour la recherche semantique du Jalon 2.**
- **gradio : le framework qui va generer l'interface web et l'URL publique du Jalon 4.**
- **pypdf : la librairie qui permet de lire le contenu texte d'un fichier PDF.**

In [15]:
%pip install -q sentence-transformers gradio pypdf
print("✅ Installation terminée !") # Message de confirmation visuel

Note: you may need to restart the kernel to use updated packages.
✅ Installation terminée !


## DETECTION DU GPU POUR LE MODELE

In [16]:
import torch

if torch.cuda.is_available():
    MODELE = "Qwen/Qwen2.5-1.5B-Instruct"
    DEVICE = 0
    print("🚀 GPU détecté : on utilise le moteur rapide.")
else:
    MODELE = "Qwen/Qwen2.5-0.5B-Instruct"
    DEVICE = -1
    print("🐢 Pas de GPU : on utilise le moteur léger. Tout marchera quand même.")

🐢 Pas de GPU : on utilise le moteur léger. Tout marchera quand même.


### En savoir plus

- `PyTorch` est la librairie de calcul numerique et de reseaux de neurones sur laquelle reposent aussi bien **sentence-transformers** que le modele de langage charge plus tard. On l'importe ici uniquement pour une chose : **detecter si une carte graphique compatible est disponible**.
  
- `torch.cuda.is_available()` renvoie True si PyTorch detecte un **GPU NVIDIA(technologie CUDA) utilisable dans l'environnement**. C'est un test d'adaptation automatique : le notebook s'ajuste au materiel reellement disponible, sans que l'utilisateur est besoin de modifier quoi que ce soit.

- Si un **GPU est present**, on choisit le modele `Qwen2.5 a 1,5 milliard de parametres` (version « Instruct », c'est-a-dire specialement entrainee pour suivre des instructions et dialoguer, contrairement a une version brute). **DEVICE = 0 indique a la librairie transformers d'utiliser le premier GPU disponible (numerote a partir de 0) pour faire tourner ce modele**. A defaut de GPU, on bascule sur une version plus legere du meme modele, avec seulement `0,5 milliard de parametres` : moins precise mais bien plus rapide a faire tourner sur un simple processeur (CPU).**DEVICE = -1 est la convention de la librairie transformers pour dire « utilise le CPU, pas de GPU »**

# JALON 1 : RECUPERATION ET DECOUPAGE DES DOCUMENTS

Un assistant IA d'entreprise répond à partir de **documents**c, pas d'une intuition.

Pourquoi du fictif ? **Parce que l'IA ne peut rien en savoir**. Si votre assistant répond juste, vous saurez que c'est grâce à **VOTRE travail, pas à sa mémoire.**

### En savoir plus sur le contenu des documents

Sur quoi repose cet assistant ?

Cet assistant s'appuie sur quatre documents PDF décrivant une entreprise fictive, la Librairie du Savoir, un réseau togolais de librairies et papeteries implanté à Lomé, Kara et Sokodé. Chaque agence dispose de son propre guide pratique (adresse, horaires, rayons, services, tarifs), et un quatrième document rassemble les conditions générales de vente communes au réseau.

Cette entreprise n'existe pas réellement : c'est volontaire. Cela garantit que toute réponse correcte de l'assistant provient forcément des documents fournis, et non d'une connaissance que le modèle aurait déjà mémorisée pendant son entraînement. Si l'assistant répond juste à une question comme « Est-ce que je peux payer par carte à Sokodé ? », c'est la preuve que le système RAG fonctionne : il retrouve puis exploite la bonne information, plutôt que de l'inventer.

Les documents sont volontairement construits avec des informations qui diffèrent d'une agence à l'autre (tarifs, services disponibles, horaires), afin de vérifier que le moteur de recherche sémantique retrouve bien les passages du bon site en fonction de la question posée.

In [17]:
import io, requests
from pypdf import PdfReader

# ⬇️ L'adresse publique où sont rangés les PDF (dataset Hugging Face) ⬇️
BASE_URL = "https://huggingface.co/datasets/wilfriedx20/Atelier-Assistant-IA/resolve/main/"
FICHIERS = ["librairie-savoir-conditions-generales.pdf", "librairie-savoir-kara.pdf",
            "librairie-savoir-lome.pdf", "librairie-savoir-sokode.pdf"]

TEXTES = {} # On cree un dictionnaire vide qui va associer, a chaque nom de fichier, le texte brut qui en a ete extrait

try:
    for f in FICHIERS:
        r = requests.get(BASE_URL + f, timeout=30) # Construction de l'URL base+nom dans une durée de 30s
        r.raise_for_status()
        pages = PdfReader(io.BytesIO(r.content)).pages
        TEXTES[f] = "\n".join((p.extract_text() or "") for p in pages)
        print(f"✅ {f:42s} {len(pages)} page(s) · {len(TEXTES[f]):5d} caractères")
except Exception as e: # Si le telechargement echoue
    import base64
    print("⚠️ Téléchargement impossible (", e, ")")
    print("   On bascule sur les six PDF de secours embarqués : la suite est identique.\n")
    TEXTES = {}
    for f, contenu_b64 in PDF_SECOURS.items():
        pages = PdfReader(io.BytesIO(base64.b64decode(contenu_b64))).pages
        TEXTES[f] = "\n".join((p.extract_text() or "") for p in pages)
        print(f"🛟 {f:42s} {len(pages)} page(s) · {len(TEXTES[f]):5d} caractères  (secours)")

print(f"\n📚 {len(TEXTES)} documents chargés.")

✅ librairie-savoir-conditions-generales.pdf  2 page(s) ·  4902 caractères
✅ librairie-savoir-kara.pdf                  2 page(s) ·  3453 caractères
✅ librairie-savoir-lome.pdf                  2 page(s) ·  4524 caractères
✅ librairie-savoir-sokode.pdf                2 page(s) ·  3371 caractères

📚 4 documents chargés.


## LE CHUCKING : DECOUPAGE

On ne va pas donner un document entier au modèle. Deux raisons :

1. **Sa fenêtre de contexte est finie** : on ne peut pas tout y mettre.
2. **Un passage court se retrouve mieux qu'un document entier.**

Découper un texte en passages, c'est le **chunking**. Le nôtre est volontairement simple : des passages d'environ **500 caractères**, coupés de préférence en fin de phrase, avec un petit **chevauchement** pour ne pas perdre une information à cheval sur deux passages. En production, on fait plus sophistiqué.

Nos quatre guides font une quinzaine de pages à eux tous. Un modèle local s'y perd, c'est lent, et en entreprise on n'a pas 4 documents mais 4 000. On lui donne trois passages, les bons.

Une astuce de métier : on **encode le titre du document avec le passage**. Ainsi le moteur sait que « photocopie à 25 francs CFA la page A4 » parle de Kara, pas de Lomé. En revanche on donne au modèle le passage **seul** : si on lui colle le titre dedans, il le recopie dans sa réponse. Détail minuscule, effet immédiat.

In [18]:
def decouper(texte, taille=500, chevauchement=80):
    """Découpe un texte en passages d'environ `taille` caractères, en coupant de préférence en fin de phrase."""
    texte = " ".join(texte.split())          # on nettoie les sauts de ligne du PDF
    passages, debut = [], 0 # Passages va accueillir les morceaux de texte decoupe
    while debut < len(texte): # Tant qu'il reste du texte a decouper
        fin = min(debut + taille, len(texte)) # min evite de depasser la longueur reelle du texte
        if fin < len(texte):
            coupe = texte.rfind(". ", debut + taille // 2, fin)   # la fin de phrase la plus proche
            if coupe != -1:
                fin = coupe + 1
        passages.append(texte[debut:fin].strip()) # extraction entre debut et fin
        if fin >= len(texte): # Logique
            break
        debut = max(fin - chevauchement, debut + 1)   # on recule un peu : le chevauchement
    return [p for p in passages if p] # On renvoie la liste des passages en filtrant ceux qui seraient vides


DOCUMENTS = [] # Liste finale qui va contenir tous les passages de tous les documents

for f, t in TEXTES.items(): # f--> fichier , t-->texte
    nom = f.replace(".pdf", "").replace("librairie-savoir-", "Librairie Savoir ").replace("-", " ").title()
    for i, p in enumerate(decouper(t), 1):
        DOCUMENTS.append({"titre": f"{nom} · passage {i}", "texte": p})


print(f"✅ {len(DOCUMENTS)} passages, prêts à être encodés.")
print()
print("Exemple ·", DOCUMENTS[0]["titre"])
print(DOCUMENTS[0]["texte"][:350], "...")

✅ 46 passages, prêts à être encodés.

Exemple · Librairie Savoir Conditions Generales · passage 1
LIBRAIRIE DU SAVOIR Conditions Generales - Reference LDS-CGV-01 Document interne - reseau de librairies et papeteries (usage pedagogique) Page 1 GUIDE PRATIQUE DU RESEAU Librairie du Savoir - Conditions Generales de VenteConditions applicables a l'ensemble des agences du reseau Version mise a jour - document contractuel Article 1 - Champ d'applicat ...


### En savoir plus

`POURQUOI NE PAS COUPER "BRUTALEMENT" TOUS LES 500 CARACTÈRES ?`

Une coupe brutale risquerait de trancher une phrase en deux, par exemple : « La photocopie noir et blanc coûte 25 francs CFA la page A4, et 50 francs CFA » d'un côté et « en format A3. » de l'autre. Si le moteur de recherche ne retrouve que le premier passage, l'information est incomplète et trompeuse. Chercher le dernier point avant la limite (avec rfind) est une heuristique simple mais efficace pour garder des passages qui ont un sens complet.

**Cette methode de chunking est dite « a taille fixe avec chevauchement » : elle est simple, rapide, et suffisante pour un atelier. En production, on utilise souvent des approches plus fines, comme le decoupage par structure du document (titres, paragraphes, sections), ou un chunking « semantique » qui regroupe des phrases proches par le sens plutot que par un simple compte de caracteres. Des librairies comme `langchain` proposent des TextSplitter prets a l'emploi pour cela.** 

# JALON 2 : LE MOTEUR QUI COMPREND LE SENS

Comment retrouver le bon passage quand quelqu'un pose une question ? Pas avec des mots-clés : avec des **embeddings**.

Le principe : chaque passage est transformé en un vecteur, une liste de nombres qui capture son **sens**. La question subit le même sort. Il ne reste qu'à comparer les vecteurs : les plus proches sont les passages qui parlent de la même chose.

In [19]:
from sentence_transformers import SentenceTransformer
import numpy as np # Pour le calcul de vecteurs

# Choix du modele
encodeur = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3245.68it/s]


**On encode le TITRE avec le passage : le moteur saura ainsi de quel club on parle. Mais le texte qu'on donnera au modèle, lui, restera propre.**

In [20]:
textes = [f"{d['titre']} - {d['texte']}" for d in DOCUMENTS]

# Appel de la methode encode qui transforme, en une seule fois et de maniere optimisee,
# toute la liste de textes en une matrice de vecteurs numeriques
vecteurs = encodeur.encode(textes, normalize_embeddings=True) 

print(f"✅ {len(vecteurs)} passages encodés.") # Confirmation
print(f"Chaque passage est devenu un vecteur de {vecteurs.shape[1]} nombres.")
#  Une propriete fixe du modele d'embedding choisi, generalement 384 pour ce modele MiniLM)

✅ 46 passages encodés.
Chaque passage est devenu un vecteur de 384 nombres.


### En savoir plus

`POURQUOI ENCODER LE TITRE AVEC LE PASSAGE ?`

Si on encode uniquement le texte brut « ouvert du lundi au samedi de 8h00 à 18h30 », le moteur ne saura pas de quelle agence il s'agit : cette phrase pourrait appartenir à n'importe laquelle des trois villes. En intégrant le titre (« Librairie du Savoir - Agence de Kara ») dans le texte qui est transformé en vecteur, on donne au moteur de recherche une information de contexte cruciale, qui améliore nettement la pertinence des résultats quand la question mentionne une ville précise.



**Un embedding n'est rien d'autre qu'un point dans un espace a plusieurs centaines de dimensions. Deux textes qui « veulent dire la meme chose » se retrouvent proches dans cet espace, meme s'ils n'ont presque aucun mot en commun. C'est ce mecanisme, invente pour le traitement du langage, qui est egalement au coeur des systemes de recommandation, de la recherche d'images similaires, ou du classement de documents.**

In [21]:
# Combien de passages on donne au modèle ?
def chercher(question, k=3): # Fonction de recherche semantique
    """Renvoie les k passages les plus proches de la question."""
    v_question = encodeur.encode(question, normalize_embeddings=True) # Question transformee en vecteur
    similarites = vecteurs @ v_question # Multiplication matricielle :)
    indices = np.argsort(-similarites)[:k] # Rangement  du proche au moins proche d'ou le moins
    return [(DOCUMENTS[i]["titre"], DOCUMENTS[i]["texte"], float(similarites[i]))
            for i in indices]

# La question peut etre poser dans les parentheses
# Avoir differentes villes n'est pas une erreur en soi
for titre, texte, score in chercher("Le tarif a Lome?"):
    print(f"[{score:.2f}] {titre}")

print()
print("✅ Le moteur retrouve les bons passages, et la bonne ville !")

[0.48] Librairie Savoir Lome · passage 1
[0.45] Librairie Savoir Lome · passage 2
[0.44] Librairie Savoir Lome · passage 13

✅ Le moteur retrouve les bons passages, et la bonne ville !


`POURQUOI CHOISIR "COMBIEN DE PASSAGES" (K) EST IMPORTANT ?`

Un k trop petit (par exemple 1) risque de rater la bonne information si elle est repartie sur plusieurs passages, ou si le tout premier resultat n'est pas exactement le bon. Un k trop grand (par exemple 20) noie le modele de langage sous une quantite d'information excessive, dont une bonne partie n'est pas pertinente, ce qui peut degrader la qualite de sa reponse et ralentir le traitement. La valeur  k=3 est un compromis courant pour un premier prototype : assez de matiere pour couvrir la reponse, assez peu pour rester concis et focalise.

## La preuve que c'est du sens, pas des mots

Posons une question dont **aucun mot-clé** n'apparaît dans le bon passage : « Ce livre ne me plaît pas finalement, puis-je le rendre au magasin ? ». Le passage qui répond parle d'« échange » et d'« avoir », jamais de « rendre ». Regardez.

In [22]:
for titre, texte, score in chercher("Ce livre ne me plaît pas finalement, puis-je le rendre au magasin ?"):
    print(f"[{score:.2f}] {titre}")

[0.38] Librairie Savoir Conditions Generales · passage 6
[0.37] Librairie Savoir Conditions Generales · passage 8
[0.36] Librairie Savoir Conditions Generales · passage 4


### En savoir plus

Une recherche lexicale sur « rendre » ne trouverait rien dans un document qui parle de «
echange », meme si c'est precisement la reponse a la question. C'est cette limite que les embeddings permettent de depasser.

# JALON 3 : L'IA qui redige

Il nous manque la troisième brique : le modèle de langage qui va rédiger les réponses.

On va le charger, puis faire une expérience en deux temps:

1. On va lui poser une question **sans** lui donner le moindre document.
2. **Ensuite** on lui posera exactement la même, avec les documents. Et on affichera les deux réponses l'une sous l'autre.

Choisissez une question dont la réponse est **vérifiable** dans nos guides. On saura ainsi, sans discussion possible, si la réponse est juste ou non.

In [24]:
# Combien coûte la plastification d'un document à l'agence de Sokodé ?

In [25]:
from transformers import pipeline
import transformers

transformers.logging.set_verbosity_error()   # on masque les avertissements techniques

print("Chargement du modèle (1 à 3 minutes la première fois)...")
generateur = pipeline("text-generation", model=MODELE, device=DEVICE)

# On règle la génération une fois pour toutes
generateur.tokenizer.clean_up_tokenization_spaces = False # Permet d'eliminer les espaces entre les points d'interrogation et certains mots
generateur.model.generation_config.max_new_tokens = 180 # IMPORTANT : Ca indique que le modele va s'arreter s'il atteint la limite de token
generateur.model.generation_config.do_sample = False # Prendre une reponse au hasard

# En mettant oui sur le sample les autres config doivent etre modifier
# Il apporte de la temperature = creativite, humour
generateur.model.generation_config.temperature = None
generateur.model.generation_config.top_p = None
generateur.model.generation_config.top_k = None

print("✅ Modèle chargé !")


Chargement du modèle (1 à 3 minutes la première fois)...


C:\Users\HP ELITEBOOK\anaconda3\envs\assistant-ia\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP ELITEBOOK\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 3017.57it/s]


✅ Modèle chargé !


### En savoir plus 

La temperature, le top_p (nucleus sampling) et le top_k sont des reglages classiques des modeles de langage generatifs. Une temperature elevee (par exemple 1,2) rend les reponses plus variees et parfois surprenantes ; une temperature basse (proche de 0) les rend plus predictibles. Le top_p limite le choix du mot suivant aux options qui cumulent une certaine probabilite (par exemple 90 %), et le top_k le limite aux K options les plus probables. Ces trois leviers sont utiles pour des usages creatifs (ecriture de fiction, brainstorming) mais contre-productifs pour un usage factuel comme celui de cet atelier.

In [29]:
def demander_au_modele(messages):
    sortie = generateur(messages)
    return sortie[0]["generated_text"][-1]["content"]

In [30]:
# Prompting
# Dans toute vraie application, l'assistant a un RÔLE. Le nôtre :
ROLE = ("Tu es l'assistant de la Librairie du Savoir, un réseau de librairies et papeteries au Togo. "
        "Tu réponds aux questions des clients de façon précise, utile et directe.")

`POURQUOI UN PROMPT SYSTEME SEPARE DE LA QUESTION ?`

Separer le « role » (qui reste stable, quelle que soit la question posee) de la « question » elle-meme (qui change a chaque appel) est une pratique standard du prompting. Cela permet de configurer une bonne fois pour toutes le comportement general de l'assistant, et de le reutiliser telle quelle pour toutes les questions suivantes, sans avoir a la reecrire a chaque fois.

In [33]:
# votre question : sa réponse est dans nos guides, mais le modèle ne les a pas encore.
question_test = "Combien coûte la location d'un document à l'agence de Sokodé ?"

# On garde cette réponse de côté : on la comparera tout à l'heure.
reponse_sans_documents = demander_au_modele([
    {"role": "system", "content": ROLE},          # il est l'assistant du club...
    {"role": "user", "content": question_test},   # ...mais il n'a aucun document.
])
print(reponse_sans_documents)

Je vous suggère de consulter directement l'agence de Sokodé ou son site web pour obtenir le prix exact de la location d'un document. Ils sont en mesure de fournir des informations précises sur les frais de location et de taxes liées à cette action.


### Que vient-il de se passer ?

La Librairie du Savoir **n'existe pas**. Le modèle n'avait donc aucun moyen de connaître la réponse. Et pourtant il a produit quelque chose.

**Deux choses différentes peuvent se produire.**

- Chez certains, il a inventé une réponse. Un prix, un horaire, avec assurance. C'est une **hallucination** : quand un modèle ne sait pas, il complète, parce que compléter est tout ce qu'il sait faire.
- Chez d'autres, il a refusé : « je n'ai pas cette information, contactez le magasin ».

Une seule cause aux deux pannes : **il n'a pas nos documents**.

Regardez bien la cellule suivante : **le rôle ne change pas**. Ce sont les documents et la règle qui s'ajoutent.

In [34]:
def repondre(question):
    """L'assistant complet : recherche + rédaction sous contrainte."""
    passages = chercher(question) # on recherches les K passages les plus pertinents vis-a-vis de la question
    contexte = "\n\n".join(f"### {t}\n{x}"for t, x, _ in passages) # on mets tous ces "passages" au sein d'une meme string, ca sera notre "contexte"

    # On Construit le Prompt Final : qu'est-ce qu'on lui interdit ? et que doit-il dire s'il ne sait pas ?
    messages = [
        {"role": "system", "content": ROLE +          # <- le MÊME rôle que tout à l'heure
            " Tu réponds UNIQUEMENT à partir des documents fournis. "
            "Si la réponse n'y figure pas, réponds exactement : « Cette information ne figure pas dans nos documents. Veuillez contacter le magasin ou notre service client. »"},
        {"role": "user", "content": f"Documents :\n{contexte}\n\nQuestion : {question}"},
    ]

    reponse = demander_au_modele(messages)
    sources = ", ".join(t for t, _, _ in passages)
    return reponse, sources


# La MÊME question, cette fois avec les documents. On affiche les deux réponses l'une sous l'autre.
reponse_avec_documents, sources = repondre(question_test)

print("❓", question_test)
print()
print("❌ SANS les documents :")
print("   ", " ".join(reponse_sans_documents.split())[:400])
print()
print("✅ AVEC les documents :")
print("   ", " ".join(reponse_avec_documents.split()))
print("    📎 Sources :", sources)

❓ Combien coûte la location d'un document à l'agence de Sokodé ?

❌ SANS les documents :
    Je vous suggère de consulter directement l'agence de Sokodé ou son site web pour obtenir le prix exact de la location d'un document. Ils sont en mesure de fournir des informations précises sur les frais de location et de taxes liées à cette action.

✅ AVEC les documents :
    La location d'un document à l'agence de Sokode est possible et gratuite. Le coût est de 25 francs CFA pour une page A4.
    📎 Sources : Librairie Savoir Sokode · passage 5, Librairie Savoir Sokode · passage 7, Librairie Savoir Sokode · passage 2


In [35]:
questions = [
    "L'agence de Kara propose-t-elle un rayon universitaire ?",
    "Quelle agence propose un service de plastification de documents ?",
    "L'agence de Sokodé est-elle ouverte le dimanche pendant la période de rentrée scolaire ?",
    "Combien coûte l'impression couleur à l'agence de Lomé ?",
    "Quel est le délai pour échanger un ouvrage après achat ?",
    "Quel est le cours du Bitcoin ?",
]

for q in questions:
    r, s = repondre(q)
    print(f"❓ {q}")
    print(f"💬 {r}")
    print(f"📎 {s}")
    print()

print("🎉Notre assistant vient de répondre !")

❓ L'agence de Kara propose-t-elle un rayon universitaire ?
💬 La librairie du Savoir de Kara offre un rayon universitaire spécifiquement adapté à l'école primaire et l'école de terminale. Cela signifie que les ouvrages pratiques peuvent être disponibles sur ce point. Cependant, il existe des rayons littéraires généraux qui sont également disponibles à l'agence de Lome.
📎 Librairie Savoir Kara · passage 10, Librairie Savoir Kara · passage 1, Librairie Savoir Kara · passage 4

❓ Quelle agence propose un service de plastification de documents ?
💬 La service de plastification de documents est proposée à Sokode par une coûte de 300 francs CFA.
📎 Librairie Savoir Kara · passage 5, Librairie Savoir Sokode · passage 5, Librairie Savoir Lome · passage 6

❓ L'agence de Sokodé est-elle ouverte le dimanche pendant la période de rentrée scolaire ?
💬 La réponse à votre question est : Non, la librairie de Sokodé ne fait pas mentionner de période de rentrée scolaire pour ses heures d'ouverture. Les inf

Notez la dernière question : le Bitcoin n'est pas dans les documents, et l'assistant **le dit** au lieu d'inventer. C'est exactement la différence entre utiliser l'IA et la maîtriser.

Et remarquez ce qui a changé entre les deux réponses de tout à l'heure : **pas le modèle**, il est identique. **Pas le rôle**, il est identique. Ce sont les trois passages et une règle de huit mots. C'est tout le RAG.

# JALON 4 : LA MISE EN LIGNE 

Notre assistant fonctionne. Mais il vit dans ce notebook, sur cette page. Personne d'autre que moi ne peut s'en servir.

On va lui donner deux choses : un **visage** (une vraie page web) et une **adresse publique**. Un seul mot crée un tunnel : votre application, qui tourne ici, devient accessible depuis n'importe où dans le monde.

In [36]:
import gradio as gr

def assistant_web(question):
    if not question.strip():
        return "Posez-moi une question sur les Librairies Savoir!"
    reponse, sources = repondre(question)
    return f"{reponse}\n\n📎 Sources : {sources}"

demo = gr.Interface(
    fn=assistant_web,
    inputs=gr.Textbox(label="Votre question",
                      placeholder="Ex : le club de Lyon a-t-il une piscine ?"),
    outputs=gr.Textbox(label="Réponse de l'assistant"),
    title="Mon premier assistant IA 🤖",
    description="Il répond aux questions sur les agences de la Librairie du Savoir, un réseau de librairies et papeteries fictif "
    "(Lomé, Kara, Sokodé).Construit en direct avec Machine Learnia.",
)

# Un seul mot pour passer de notre écran au monde
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


### En savoir plus

`POURQUOI SEPARER "ASSISTANT_WEB" DE "REPONDRE" PLUTOT QUE TOUT REECRIRE ?`

Cela illustre un principe de conception solide : la logique metier (rechercher les bons passages, generer une reponse contrainte) est totalement independante de l'interface qui l'expose. La meme fonction repondre pourrait tout aussi bien etre appelee depuis une ligne de commande, une API, un bot Slack ou, comme ici, une page web Gradio, sans avoir a la modifier. Seule une fine couche d'adaptation (assistant_web) fait le lien entre les deux mondes.

**Le lien genere par share=True est volontairement temporaire (il expire au bout de quelques heures) et repose sur l'infrastructure gratuite de Gradio : c'est parfait pour une demonstration ou un atelier, mais pas pour une mise en production durable. Pour une application destinee a rester en ligne durablement, on heberge generalement l'application sur un service dedie (par exemple Hugging Face Spaces, un serveur cloud, ou une plateforme de deploiement continue), avec un nom de domaine stable.**